<a href="https://colab.research.google.com/github/ronan-cunha/machine-learning-course/blob/main/Notebooks/aula_14_xgboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml, load_diabetes
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

In [7]:
# ==============================================================================
# ETAPA 1: Carregar Dataset Real (Diabetes)
# ==============================================================================
# O dataset possui 442 observações e 10 variáveis preditoras clínicas:
# age (idade), sex (sexo), bmi (IMC), bp (pressão arterial média)
# s1 a s6 (seis medições do soro sanguíneo, ex: colesterol total, LDL, HDL, triglicérides, glicose)
# https://scikit-learn.org/stable/datasets/toy_dataset.html#diabetes-dataset
diabetes = load_diabetes(as_frame=True)
df = diabetes.frame
df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [8]:
# ==============================================================================
# ETAPA 2: Divisão entre Treino e Teste
# ==============================================================================
X = df.drop(columns=["target"])
y = df["target"]

feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Amostras de Treino: {X_train.shape[0]} | Amostras de Teste: {X_test.shape[0]}\n")

Amostras de Treino: 353 | Amostras de Teste: 89



In [9]:
# ==============================================================================
# ETAPA 3: Otimização de Hiperparâmetros (XGBoost Regressor)
# ==============================================================================

# Definição do espaço de hiperparâmetros para busca no XGBoost Regressor
param_grid = {
    'n_estimators': [100, 250],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5, 8],
    'subsample': [0.7, 1.0]
}

# Instanciando o modelo base para regressão
xgb_base = XGBRegressor(random_state=42, n_jobs=-1, eval_metric='rmse')

In [10]:
# Configurando o Grid Search com 5-fold cross-validation usando métrica de regressão
#sklearn.metrics.get_scorer_names()
grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

print("Iniciando Grid Search...")
grid_search.fit(X_train, y_train)

Iniciando Grid Search...
Fitting 5 folds for each of 24 candidates, totalling 120 fits


GridSearchCV(cv=5,
             estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=True, eval_metric='rmse',
                                    feature_types=None, feature_weights=None,
                                    gamma=None, grow_policy=None,
                                    importance_type=None,
                                    interaction_constraints=Non...
                                    max_cat_to_onehot=None, max_delta_step=None,
                                    max_depth=None, max_leaves=None,
                                    min_child_weight=None, missing=nan,
                                    monotone_constraints=None,
                                    multi_strategy=None, n_estimators=None,
                                    n_jobs=-1, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'learning_rate': [0.01, 0.1], 'max_depth': [3, 5, 8],
                         'n_estimators': [100, 250], 'subsample': [0.7, 1.0]},
             scoring='neg_mean_squared_error', verbose=1)

In [11]:
# Melhores parâmetros e resultado do Grid Search
best_xgb_grid = grid_search.best_estimator_
y_pred_grid = best_xgb_grid.predict(X_test)
mse = mean_squared_error(y_test, y_pred_grid)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred_grid)

print(f"\nMelhores Parâmetros (Grid Search): {grid_search.best_params_}")

print("--- Métricas de Avaliação no Teste ---")
print(f"MAE  (Erro Médio Absoluto): {mae:.2f}")
print(f"RMSE (Raiz do Erro Quadrático Médio): {rmse:.2f}")


Melhores Parâmetros (Grid Search): {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 250, 'subsample': 0.7}
--- Métricas de Avaliação no Teste ---
MAE  (Erro Médio Absoluto): 43.08
RMSE (Raiz do Erro Quadrático Médio): 52.32


In [12]:
# Configurando o Random Search (testando 20 combinações aleatórias) para regressão
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Iniciando Random Search...")
random_search.fit(X_train, y_train)

Iniciando Random Search...
Fitting 5 folds for each of 20 candidates, totalling 100 fits


RandomizedSearchCV(cv=5,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=True,
                                          eval_metric='rmse',
                                          feature_types=None,
                                          feature_weights=None, gamma=None,
                                          grow_policy=None,
                                          importance_type=None,
                                          interaction_constrain...
                                          max_delta_step=None, max_depth=None,
                                          max_leaves=None,
                                          min_child_weight=None, missing=nan,
                                          monotone_constraints=None,
                                          multi_strategy=None,
                                          n_estimators=None, n_jobs=-1,
                                          num_parallel_tree=None, ...),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'learning_rate': [0.01, 0.1],
                                        'max_depth': [3, 5, 8],
                                        'n_estimators': [100, 250],
                                        'subsample': [0.7, 1.0]},
                   random_state=42, scoring='neg_mean_squared_error',
                   verbose=1)

In [17]:
# Obtendo previsões do melhor modelo do Grid Search para comparação
best_xgb_grid = grid_search.best_estimator_
y_pred_grid = best_xgb_grid.predict(X_test)
mse_random = mean_squared_error(y_test, y_pred_grid)
rmse_random = np.sqrt(mse_grid)
mae_random = mean_absolute_error(y_test, y_pred_grid)

print("--- Comparação de Desempenho dos Modelos (RMSE - menor é melhor) ---")
print(f"Grid Search Tuning  - RMSE: {rmse_grid:.2f} | MAE: {mae_grid:.2f}")
print(f"Random Search Tuning- RMSE: {rmse_random:.2f} | MAE: {mae_random:.2f}")

--- Comparação de Desempenho dos Modelos (RMSE - menor é melhor) ---
Grid Search Tuning  - RMSE: 52.32 | MAE: 43.08
Random Search Tuning- RMSE: 52.32 | MAE: 43.08


# Atividade

Usando a base de dados dos imóveis da Califórina, responda as questões abaixo:

- from sklearn.datasets import fetch_california_housing
- housing = fetch_california_housing()
- df_real = pd.DataFrame(housing.data, columns=housing.feature_names)
- df_real['MEDV'] = housing.target

a) Separe 20% para a amostra de teste;

b) Treine uma floresta aleatória, selecionando de forma ótima os hiperparâmetros.

c) Treine um XGBoost, selecionando de forma ótima os hiperparâmetros.

d) Justifique as escolhas dos valores testados (grid de parâmetros), o critério de busca utilizado (grid search ou random search) e a métrica de desempenho escolhida.

e) Compare o desempenho dos dois modelos. Qual teve o melhor desempenho na amostra de teste?